# 01. Przegląd zbiorów danych i wybór tematu

**Krok procesu:** 1. Wybór tematu na podstawie przeglądu dostępnych zbiorów danych (zob. `docs/proces_projektowy.png`).

**Temat:** liczenie palców w czasie rzeczywistym. Kamera telefonu widzi dłoń, a aplikacja pokazuje, ile palców
jest teraz wyprostowanych.

**Wymagania wobec zbioru:** dużo zdjęć pojedynczych dłoni z różnymi układami palców, różne osoby, tła
i oświetlenie. Licencja pozwalająca na użycie akademickie. Najlepiej z ramkami dłoni, żeby dało się wyciąć dłoń.

## Kandydaci: HaGRID v1 vs HaGRIDv2

Dane z oficjalnego repozytorium (github.com/hukenovs/hagrid), sprawdzone 2026-10-04.

| | HaGRID v1 | HaGRIDv2 |
|---|---|---|
| Obrazy | 552,992 FullHD | 1,086,158 FullHD |
| Klasy gestów | 18 | 33 + no_gesture |
| Unikalne osoby | 34,730 | 65,977 |
| Pełny rozmiar | 716 GB | 1.5 TB |
| Wersja pomniejszona | brak | 512 px, 119.4 GB |
| Podział | 92/8 train/test | 76/9/15 train/val/test |

Obie wersje mają ramki dłoni (format COCO), punkty charakterystyczne dłoni z MediaPipe (landmarki, 21 punktów
na dłoń) i metadane (wiek, płeć, rasa). Licencja to wariant CC BY-SA 4.0.

**Wybieramy HaGRIDv2.** Dwa razy więcej obrazów i osób, więcej gestów jednej dłoni, które da się przypisać do liczby
palców (np. `three2`, `three3`, `three_gun`, `thumb_index`, `little_finger`, `middle_finger`), osobny zbiór
walidacyjny i wersja 512 px, która mieści się na dysku laptopa.

**Haczyk:** żaden zbiór nie ma etykiety "liczba palców". Sami przypisujemy klasom gestów liczbę palców
(zob. `src/labels.py`) i sprawdzamy to mapowanie w `02_eda.ipynb`.

In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from labels import DROPPED, GESTURE_TO_COUNT, NO_GESTURE

images = pd.read_parquet("../data/interim/images.parquet")
hands = pd.read_parquet("../data/interim/hands.parquet", columns=["split", "label", "user_id"])

## Porównanie opublikowanych liczb z pobranymi adnotacjami

Pobraliśmy tylko archiwum z adnotacjami (zip 719 MB, 4 GB plików JSON). Skrypt `src/build_annotations_table.py`
spłaszcza je do `data/interim/images.parquet` i `data/interim/hands.parquet`.

In [2]:
summary = pd.Series({
    "obrazy": len(images),
    "dłonie z adnotacją": len(hands),
    "unikalne osoby (user_id)": images.user_id.nunique(),
    "klasy gestów (bez no_gesture)": hands.label[hands.label != NO_GESTURE].nunique(),
})
summary.to_frame("wartość")

,wartość
obrazy,1086158
dłonie z adnotacją,1504133
unikalne osoby (user_id),65976
klasy gestów (bez no_gesture),33


In [3]:
split_share = images.split.value_counts(normalize=True).mul(100).round(1)
split_share.to_frame("% obrazów")

,% obrazów
split,
train,75.6
test,15.2
val,9.1


## Co zostawiamy do liczenia palców

In [4]:
mapping = (pd.Series(GESTURE_TO_COUNT, name="palce").rename_axis("gest").reset_index()
           .sort_values(["palce", "gest"]))
mapping.groupby("palce")["gest"].apply(", ".join).to_frame("gesty źródłowe")

,gesty źródłowe
palce,
0,fist
1,"dislike, like, little_finger, middle_finger, m..."
2,"call, peace, peace_inverted, rock, thumb_index..."
3,"ok, three, three2, three3, three_gun"
4,four
5,"palm, stop, stop_inverted"


In [5]:
pd.DataFrame({"odrzucony gest": DROPPED, "powód": "dwie dłonie albo brak jednoznacznej liczby palców"})

,odrzucony gest,powód
0,hand_heart,dwie dłonie albo brak jednoznacznej liczby palców
1,hand_heart2,dwie dłonie albo brak jednoznacznej liczby palców
2,holy,dwie dłonie albo brak jednoznacznej liczby palców
3,timeout,dwie dłonie albo brak jednoznacznej liczby palców
4,xsign,dwie dłonie albo brak jednoznacznej liczby palców
5,take_picture,dwie dłonie albo brak jednoznacznej liczby palców
6,grabbing,dwie dłonie albo brak jednoznacznej liczby palców
7,grip,dwie dłonie albo brak jednoznacznej liczby palców


## Wnioski

- Archiwum adnotacji zgadza się z README: 1,086,158 obrazów i 33 klasy gestów plus `no_gesture`.
  W adnotacjach jest 65,976 różnych wartości `user_id`, o jedną mniej niż 65,977 z README.
- 25 z 33 gestów da się przypisać do liczby palców od 0 do 5. 8 odrzucamy.
- Klasa 4 pochodzi z jednego gestu (`four`), więc spodziewamy się, że będzie klasą mniejszościową. Mierzymy to
  w `02_eda.ipynb`.
- Adnotacje zawierają już landmarki z MediaPipe. Możemy więc sprawdzić model oparty na punktach charakterystycznych,
  zanim pobierzemy jakiekolwiek obrazy (`03_path_keypoints.ipynb`).